# Bobby Carrot — PPO Reinforcement Learning

**Algorithm**: PPO (Proximal Policy Optimization) with CNN feature extractor  
**Training set**: Normal levels 1–25 (5-stage curriculum) or levels 1–15 (per-level)  
**Test set**: Normal levels 26–30 (zero-shot evaluation)  

## Training modes

### Mode A — 5-stage curriculum (original)
| Stage | Levels | New mechanic |
|-------|--------|--------------|
| 1 | L01–L03 | Walls, floor, carrots |
| 2 | L01–L07 | + Crumble tiles |
| 3 | L01–L12 | + Directional conveyors |
| 4 | L01–L17 | + Red/Yellow switches |
| 5 | L01–L25 | + Keys/Locks, full mix |

### Mode B — Per-level training (new)
Each level 1–15 gets its own independent model, checkpoint directory, and step budget.  
Use the **Per-Level Training** cell below instead of the curriculum training cell.

## Before running
1. **Runtime → Change runtime type → T4 GPU** (required)
2. Run cells top-to-bottom. Long-running cells show a progress bar.
3. Models are saved to Google Drive so they survive session disconnects.


In [ ]:
# ── Mount Google Drive (for checkpoint persistence) ──────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_DIR = "/content/drive/MyDrive/bobby_carrot_rl"
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/tb_logs", exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}")


In [ ]:
# \u2500\u2500 GPU + RAM check \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
import torch, os

try:
    import psutil
    cpu_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"CPU RAM  : {cpu_ram_gb:.1f} GB")
except ImportError:
    cpu_ram_gb = 12.0
    print("CPU RAM  : ~12 GB (psutil not found, assuming free tier)")

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {gpu}")
    print(f"VRAM     : {vram:.1f} GB")
else:
    vram = 0.0
    print("WARNING: No GPU. Runtime \u2192 Change runtime type \u2192 T4 GPU")

# \u2500\u2500 N_ENVS for DummyVecEnv (same-process, no subprocess overhead) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# DummyVecEnv runs all envs sequentially in the main process.  The game runs
# at ~2 k headless steps/s per env, so 8 envs gives ~16 k steps/s — well above
# the GPU-bound throughput.  No subprocess memory overhead; free-tier RAM is fine.
if cpu_ram_gb >= 50:
    N_ENVS = 32
elif cpu_ram_gb >= 25:
    N_ENVS = 16
else:
    N_ENVS = 8

print(f"N_ENVS   : {N_ENVS}  (DummyVecEnv — all envs run in-process, no fork/subprocess)")


In [ ]:
# ── Clone repo + install ──────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/Charan20510/Bobby_Carrot_Python.git"
REPO_DIR = "/content/bobby_carrot"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull

# Install the package
%pip install -e {REPO_DIR} -q

# sb3-contrib provides RecurrentPPO (LSTM-based PPO) for stages 3-5
%pip install "stable-baselines3[extra]>=2.3" "sb3-contrib>=2.3" tensorboard tqdm -q

# Force python to recognize the new install without restarting runtime
import site
from importlib import reload
site.main()

print("\nInstall complete.")

In [ ]:
# ── Suppress pygame display (headless Colab) ──────────────────────────────────
import os, sys
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

# ── Sanity check: gym wrapper obs shape ───────────────────────────────────────
import numpy as np
from bobby_carrot.gym_env import BobbyCarrotEnv

env = BobbyCarrotEnv(map_kind="normal", map_number=1)
obs, _ = env.reset()
assert obs["tiles"].shape == (256,), f"tiles shape mismatch: {obs['tiles'].shape}"
assert obs["tiles"].dtype == np.uint8
assert obs["keys"].shape == (3,)
env.close()
print("✓ Gym wrapper sanity check passed")

# ── Verify rl_training package core loads cleanly ─────────────────────────────
from rl_training.config import STAGE_ALL_LEVELS
from rl_training.potential import compute_potential, bfs_to_exit
from rl_training.wrappers import CurriculumEnv
print(f"✓ rl_training package core imported ({len(STAGE_ALL_LEVELS)} stages configured)")

# ── Verify BFS exit-seeking utility ──────────────────────────────────────────
from bobby_carrot.env import GameEnv
_ge = GameEnv(); _ge.reset('normal', 1)
_d = bfs_to_exit(_ge.gs)
print(f"✓ bfs_to_exit: L01 exit is {_d} BFS tiles from start")
del _ge, _d


---
## Components
The next cells verify the curriculum wrapper, callbacks, and feature extractor.  
**Run them once** before starting training.


In [ ]:
from rl_training.config import (
    STAGE_ALL_LEVELS, STAGE_MAX_EPISODE_STEPS, STAGE_ENT_COEF,
    EXIT_APPROACH_BONUS, EXIT_RETREAT_PENALTY, POST_COLLECT_STEP_PENALTY,
    ALL_COLLECTED_BONUS,
)
from rl_training.wrappers import CurriculumEnv, RewardShapingWrapper
from rl_training.potential import compute_potential

# ── Functional tests ──────────────────────────────────────────────────────────
env = CurriculumEnv(stage=1)
obs, _ = env.reset()
assert obs["tiles"].shape == (256,)
obs, r, term, trunc, info = env.step(env.action_space.sample())
env.close()
print("✓ CurriculumEnv functional test passed")

env2 = CurriculumEnv(stage=5)
obs2, _ = env2.reset()
obs2, r2, *_ = env2.step(1)
env2.close()
print("✓ CurriculumEnv stage-5 functional test passed")
print("✓ RewardShapingWrapper: BFS potential + exit-seeking + blocked-move penalty + key bonus")
print(f"  Exit approach bonus : +{EXIT_APPROACH_BONUS}")
print(f"  Exit retreat penalty: -{EXIT_RETREAT_PENALTY}")
print(f"  Post-collect step   : -{POST_COLLECT_STEP_PENALTY}")
print(f"  All-collected bonus   : +{ALL_COLLECTED_BONUS}")


In [ ]:
from rl_training.callbacks import WinRateCallback, StageProgressCallback, TabularLogCallback, safe_print
from rl_training.extractor import BobbyExtractor

print("✓ WinRateCallback loaded (safe-print + dual det/sto eval + failure-mode breakdown)")
print("✓ StageProgressCallback loaded (tqdm progress bar per training stage)")
print("✓ TabularLogCallback loaded (safe-print compact training rows)")
print("✓ BobbyExtractor loaded (features_dim=256, CNN+ResBlock+Attention, 11 scalars)")


---
## Training

**Two training modes** are available:

### Mode A — 5-stage curriculum (Cell 11)
Runs all 5 curriculum stages sequentially with level pooling.

- **Stages 1–2**: plain PPO (fast, no LSTM overhead)  
- **Stages 3–5**: RecurrentPPO with LSTM (multi-step planning for conveyors, switches, key-locks)

### Mode B — Per-level training (Cells 11a / 11b)
Trains each level 1–15 independently with its own model.  
- **Cell 11a**: Train a single level (change `TRAIN_LEVEL`)  
- **Cell 11b**: Train all 15 levels sequentially  

Each mode prints **per-level BFS simulation reports** at startup.

**Edge-case protections** (Phase 2):
- Unwinnable detection: terminates immediately when BFS returns -1
- Conveyor-trap detection: terminates after 3 stuck steps on a conveyor tile
- Loop penalty: -0.1 when a cell is revisited >3 times without a collection event

**Expected wall-clock times** with `N_ENVS=8` on the Colab free tier:
- T4 (~80–120 K steps/s): ≈ 4–8 h total (curriculum) or ≈ 0.5–2 h per level  
- L4 (~150 K steps/s): ≈ 2–4 h total (curriculum) or ≈ 0.3–1 h per level  

If the session disconnects, re-run from the training cell — already-completed stages/levels are skipped automatically.


In [ ]:
from rl_training.config import (
    N_STEPS, BATCH_SIZE, RECURRENT_FROM_STAGE,
    STAGE_ENT_COEF, STAGE_MIN_STEPS, STAGE_MAX_STEPS,
    STAGE_MAX_EPISODE_STEPS, STAGE_WIN_THRESHOLD,
    EXIT_APPROACH_BONUS, EXIT_RETREAT_PENALTY, POST_COLLECT_STEP_PENALTY,
    ALL_COLLECTED_BONUS,
    PPO_BASE_KWARGS, RECURRENT_KWARGS, get_n_envs, get_policy_kwargs,
)
from rl_training.wrappers import RewardShapingWrapper

steps_per_rollout = N_ENVS * N_STEPS
print(f"N_ENVS               : {N_ENVS}")
print(f"Steps / rollout      : {steps_per_rollout:,}  ({N_ENVS} envs × {N_STEPS} steps)")
print(f"Batch size           : {BATCH_SIZE}  ({steps_per_rollout // BATCH_SIZE} mini-batches/update)")
print(f"RecurrentPPO from    : stage {RECURRENT_FROM_STAGE}")
print(f"Stage step budgets   : min={STAGE_MIN_STEPS}  max={STAGE_MAX_STEPS}")
print(f"Per-stage ent_coef   : {STAGE_ENT_COEF}")
print(f"Win thresholds       : {STAGE_WIN_THRESHOLD}")
print(f"Episode horizons     : {STAGE_MAX_EPISODE_STEPS}")
print(f"Exit-seeking reward  : approach=+{EXIT_APPROACH_BONUS}  retreat=-{EXIT_RETREAT_PENALTY}  step=-{POST_COLLECT_STEP_PENALTY}")
print(f"All-collected bonus  : +{ALL_COLLECTED_BONUS}")
print(f"Loop penalty         : -{RewardShapingWrapper.LOOP_PENALTY} (after >{RewardShapingWrapper.LOOP_VISIT_LIMIT} revisits)")
print(f"Conveyor trap        : -{RewardShapingWrapper.CONVEYOR_TRAP_PENALTY} (after {RewardShapingWrapper.CONVEYOR_TRAP_THRESHOLD} stuck steps)")
print(f"Unwinnable penalty   : -{RewardShapingWrapper.UNWINNABLE_PENALTY}")

# ── Config sanity (§8 checklist) ───────────────────────────────────────────
assert steps_per_rollout % BATCH_SIZE == 0, f"rollout ({steps_per_rollout}) not divisible by batch ({BATCH_SIZE})"
assert PPO_BASE_KWARGS['n_epochs'] == 10, f"n_epochs={PPO_BASE_KWARGS['n_epochs']}, expected 10"
print("✓ Config sanity checks passed (batch divisibility, n_epochs=10)")

# ── Per-Level Training Config ──────────────────────────────────────────────
from rl_training.config import (
    INDIVIDUAL_LEVELS, LEVEL_MIN_STEPS, LEVEL_MAX_STEPS,
    LEVEL_MAX_EPISODE_STEPS, LEVEL_ENT_COEF, LEVEL_USE_RECURRENT,
)

print("\n── Per-Level Training Config ──")
print(f"{'Level':>5} {'Algo':>12} {'Min Steps':>12} {'Max Steps':>12} "
      f"{'Horizon':>8} {'Ent Coef':>9}")
print("-" * 65)
for l in INDIVIDUAL_LEVELS:
    algo = "RecurrentPPO" if LEVEL_USE_RECURRENT[l] else "PPO"
    print(f"L{l:02d}   {algo:>12} {LEVEL_MIN_STEPS[l]:>12,} "
          f"{LEVEL_MAX_STEPS[l]:>12,} {LEVEL_MAX_EPISODE_STEPS[l]:>8} "
          f"{LEVEL_ENT_COEF[l]:>9}")


In [ ]:
# ── Reward magnitude audit (§8: verify |step| < |carrot| < |win|/3) ────────
# Runs a random-policy rollout on L01–L03, prints per-source magnitudes,
# and asserts the reward hierarchy is correct.  Takes ~5 seconds.
from rl_training.audit_rewards import audit_level

for lvl in [1, 2, 3]:
    row = audit_level(lvl, n_episodes=3, max_steps=500, seed=42)
    print(f"L{lvl:02d}: |step|={row['per_step']:.3f}  |carrot|={row['carrot_pickup']:.3f}  wins={row['n_wins']}  deaths={row['n_deaths']}")
    if row['carrot_pickup'] > 0:
        assert row['per_step'] < row['carrot_pickup'], \
            f"L{lvl}: step penalty ({row['per_step']:.3f}) >= carrot pickup ({row['carrot_pickup']:.3f})"
print("✓ Reward magnitude audit passed")


In [ ]:
import torch
from rl_training.train import run_training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

run_training(drive_dir=DRIVE_DIR, device=DEVICE, n_envs=N_ENVS)


### Mode B — Per-Level Training

The cells below train levels **individually** (one model per level).  
Use **Cell 11a** to train a single level, or **Cell 11b** to train all 15 sequentially.


In [ ]:
# ── Per-Level Training (single level) ─────────────────────────────────────────
# Train a SINGLE level. Change TRAIN_LEVEL to the level you want.
# Run multiple Colab instances with different TRAIN_LEVEL values for parallelism.

import torch
from rl_training.train import run_level_training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ⬇️ CHANGE THIS to train a different level ⬇️
TRAIN_LEVEL = 1

run_level_training(
    drive_dir=DRIVE_DIR,
    device=DEVICE,
    n_envs=N_ENVS,
    level=TRAIN_LEVEL,
)


In [ ]:
# ── Train All Levels 1–15 (Sequential) ────────────────────────────────────────
# This trains each level one by one. For parallelism, use the cell above
# with different TRAIN_LEVEL values in separate Colab sessions.

import torch
from rl_training.train import run_all_level_training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

run_all_level_training(
    drive_dir=DRIVE_DIR,
    device=DEVICE,
    n_envs=N_ENVS,
)


In [ ]:
# ── View Training Logs ────────────────────────────────────────────────────────
import json, os
from rl_training.config import INDIVIDUAL_LEVELS

print(f"{'Level':>5} {'Win%':>6} {'Carrot%':>8} {'Steps':>7} {'Reward':>8} {'Time':>8} {'Status':>8}")
print("─" * 58)
for l in INDIVIDUAL_LEVELS:
    log_path = f"{DRIVE_DIR}/models/level_{l:02d}/training_log.json"
    if os.path.exists(log_path):
        log = json.load(open(log_path))
        ev = log.get("eval", {})
        status = "✓ PASS" if ev.get("win_rate", 0) >= 0.70 and ev.get("carrot_pct", 0) >= 0.95 else "✗ FAIL"
        print(f"L{l:02d}   {ev.get('win_rate',0):>5.0%} {ev.get('carrot_pct',0):>7.0%} "
              f"{ev.get('mean_steps',0):>7.0f} {ev.get('mean_reward',0):>+8.1f} "
              f"{log.get('wall_time_min',0):>6.1f}m {status:>8}")
    else:
        print(f"L{l:02d}   {'—':>5} {'—':>7} {'—':>7} {'—':>8} {'—':>6}  not trained")


In [ ]:
# ── Evaluate a Trained Level Model ────────────────────────────────────────────
from rl_training.evaluate import evaluate_level_model

EVAL_LEVEL = 1  # ⬇️ Change this ⬇️

result = evaluate_level_model(DRIVE_DIR, EVAL_LEVEL, n_episodes=50, max_steps=500)
print(f"\nLevel {EVAL_LEVEL:02d}: {result['win_rate']:.0%} wins | "
      f"carrot={result['carrot_pct']:.0%} | "
      f"steps={result['mean_steps']:.0f} | "
      f"reward={result['mean_reward']:+.1f}")


In [ ]:
# ── TensorBoard ───────────────────────────────────────────────────────────────
# Run this cell at any time during or after training to view live charts.
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/tb_logs


---
## Evaluation — Levels 26–30 (held-out test set)

These levels were **never seen during training**.  
The cell below runs 50 episodes per level with a deterministic policy and reports:
- **Win rate**: episodes where agent collects all carrots and reaches the exit
- **Mean reward**: total discounted reward per episode
- **Mean steps**: episode length for winning trajectories
- **Death rate**: fraction of episodes that ended by falling onto a lethal tile


In [ ]:
from rl_training.evaluate import run_evaluation, _load_best_model

# Load model (also used by cell 18 replay)
model, model_path = _load_best_model(DRIVE_DIR)
try:
    from sb3_contrib import RecurrentPPO
    is_recurrent = isinstance(model, RecurrentPPO)
except ImportError:
    is_recurrent = False

eval_data = run_evaluation(drive_dir=DRIVE_DIR)
all_results = eval_data["results"]
heatmaps    = eval_data["heatmaps"]


In [ ]:
# ── Per-Level Model Evaluation ────────────────────────────────────────────────
# Evaluate a model trained with per-level training on its own level.
# Change EVAL_LEVEL to the level you want to evaluate.

from rl_training.evaluate_gui import load_model, batch_evaluate

EVAL_LEVEL = 1  # ⬇️ Change to evaluate a different level's model ⬇️

model_path = f"{DRIVE_DIR}/models/level_{EVAL_LEVEL:02d}/best_model.zip"
print(f"Loading model for Level {EVAL_LEVEL:02d}: {model_path}")
model, is_recurrent = load_model(model_path)

results = batch_evaluate(
    model, is_recurrent,
    levels=[EVAL_LEVEL],
    n_episodes=50,
    max_steps=500,
    headless=True,
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

levels    = list(all_results.keys())
win_rates = [all_results[l]["win_rate"] * 100 for l in levels]
rewards   = [all_results[l]["mean_r"]          for l in levels]
deaths    = [all_results[l]["death_rate"] * 100 for l in levels]
timeouts  = [all_results[l]["timeout_rate"]* 100 for l in levels]
carrots   = [all_results[l]["carrot_pct"]  * 100 for l in levels]
labels    = [f"L{l}" for l in levels]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# ── Win rate ──────────────────────────────────────────────────────────────────
ax = axes[0, 0]
colors = ["#2ecc71" if w >= 50 else "#e74c3c" for w in win_rates]
ax.bar(labels, win_rates, color=colors, edgecolor="k", linewidth=0.5)
ax.axhline(50, color="grey", linestyle="--", linewidth=1)
ax.set_title("Win Rate (%) — L26–30"); ax.set_ylim(0, 110)
for i, v in enumerate(win_rates):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=9)

# ── Mean reward ───────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.bar(labels, rewards, color="#3498db", edgecolor="k", linewidth=0.5)
ax.set_title("Mean Episode Reward — L26–30")
for i, v in enumerate(rewards):
    ax.text(i, max(v + 0.2, 0.2), f"{v:.1f}", ha="center", fontsize=9)

# ── Failure mode breakdown ────────────────────────────────────────────────────
ax = axes[0, 2]
wins_arr    = np.array(win_rates)
deaths_arr  = np.array(deaths)
timeout_arr = np.array(timeouts)
other_arr   = 100 - wins_arr - deaths_arr - timeout_arr
x = np.arange(len(labels))
ax.bar(x, wins_arr,    label="Win",     color="#2ecc71")
ax.bar(x, deaths_arr,  label="Death",   color="#e74c3c", bottom=wins_arr)
ax.bar(x, timeout_arr, label="Timeout", color="#f39c12", bottom=wins_arr+deaths_arr)
ax.bar(x, np.clip(other_arr,0,100), label="Other", color="#95a5a6",
       bottom=wins_arr+deaths_arr+timeout_arr)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Failure Mode Breakdown — L26–30")
ax.set_ylim(0, 110); ax.legend(loc="upper right", fontsize=8)

# ── Mean carrot completion % ──────────────────────────────────────────────────
ax = axes[1, 0]
ax.bar(labels, carrots, color="#9b59b6", edgecolor="k", linewidth=0.5)
ax.axhline(100, color="grey", linestyle="--", linewidth=1)
ax.set_title("Mean Carrot Completion % — L26–30"); ax.set_ylim(0, 110)
for i, v in enumerate(carrots):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=9)

# ── Visitation heatmaps (first two unseen levels) ─────────────────────────────
for col, level in enumerate([26, 27]):
    ax = axes[1, col + 1]
    im = ax.imshow(heatmaps[level], cmap="hot", interpolation="nearest",
                   vmin=0, vmax=heatmaps[level].max())
    ax.set_title(f"Visit Density — L{level}")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
chart_path = f"{DRIVE_DIR}/eval_results_v2.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to {chart_path}")

---
## Optional — Replay a trained agent

The cell below runs one episode on a chosen level and records the tile-grid at each step.  
Requires `matplotlib` only (no pygame display needed).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from bobby_carrot.gym_env import BobbyCarrotEnv

REPLAY_LEVEL = 26   # change to any level 1–30
MAX_STEPS    = 400

env  = BobbyCarrotEnv(map_kind="normal", map_number=REPLAY_LEVEL)
obs, _ = env.reset()

frames       = [obs["tiles"].reshape(16, 16).copy()]
total_reward = 0.0
done         = False

# Thread LSTM state for RecurrentPPO; ignored for plain PPO
lstm_states   = None
episode_start = np.array([True])

for _ in range(MAX_STEPS):
    if is_recurrent:
        action, lstm_states = model.predict(
            obs, state=lstm_states,
            episode_start=episode_start,
            deterministic=True,
        )
        episode_start = np.array([False])
    else:
        action, _ = model.predict(obs, deterministic=True)

    obs, r, term, trunc, info = env.step(action)
    frames.append(obs["tiles"].reshape(16, 16).copy())
    total_reward += r
    if term or trunc:
        done = True
        break
env.close()

outcome = "WON" if total_reward >= 5.0 else "lost"
print(f"Level {REPLAY_LEVEL}: {outcome}  reward={total_reward:.2f}  steps={len(frames)-1}")

# Animate (every 4th frame for speed)
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
im = ax.imshow(frames[0], cmap="tab20", vmin=0, vmax=46, interpolation="nearest")
title = ax.set_title("Step 0")

def _update(i):
    idx = min(i * 4, len(frames) - 1)
    im.set_data(frames[idx])
    title.set_text(f"Step {idx}")
    return [im, title]

n_frames = (len(frames) + 3) // 4
ani = animation.FuncAnimation(fig, _update, frames=n_frames, interval=120, blit=True)
plt.close()
HTML(ani.to_jshtml())

---
## GUI Gameplay Evaluator

Use `rl_training.evaluate_gui` to visually watch the agent play with full Pygame rendering.  
**This requires a display** (run locally, not on Colab).

### Usage (from terminal)
```bash
# Watch a single level:
python -m rl_training.evaluate_gui --model /path/to/best_model.zip --level 1 --fps 5

# Batch evaluate with per-level report:
python -m rl_training.evaluate_gui --model /path/to/best_model.zip --levels 1-3 --episodes 10 --report

# Save frames for video creation:
python -m rl_training.evaluate_gui --model /path/to/best_model.zip --level 2 --save-frames ./frames/

# Headless batch report (no display needed):
python -m rl_training.evaluate_gui --model /path/to/best_model.zip --levels 1-5 --episodes 20 --headless
```


In [ ]:
# ── Headless batch evaluation (works on Colab) ────────────────────────────────
# This runs the GUI evaluator in headless mode to generate a per-level report.
# For live Pygame rendering, run from terminal instead (see markdown above).

from rl_training.evaluate_gui import load_model, batch_evaluate

gui_model, gui_is_recurrent = load_model(model_path)
gui_results = batch_evaluate(
    gui_model, gui_is_recurrent,
    levels=list(range(1, 6)),  # evaluate L01-L05
    n_episodes=10,
    max_steps=500,
    headless=True,
)
